# Día 6 · Unidad 6 — Código profesional (III): logging

**Objetivos de aprendizaje**
- Entender la diferencia entre `print()` y un sistema de logging, y por qué el segundo es la herramienta correcta en código que se ejecuta sin supervisión.
- Configurar el módulo estándar `logging` (`basicConfig`, niveles, formato).
- Usar `logging.getLogger(__name__)` en vez del logger raíz, como se hace en proyectos reales.
- Añadir un `Handler` de archivo para persistir los logs, además del de consola.
- Registrar excepciones capturadas con contexto suficiente para depurarlas después.

**Duración estimada:** 60-75 minutos.

**Requisitos previos:** `02_teoria_excepciones.ipynb` de este mismo día.

## 1. Motivación

*Logging* es la acción de dejar constancia, en algún sitio persistente, de los eventos que ocurren durante la ejecución de un programa. Un buen sistema de logs ayuda a:

- **Depurar errores**: saber qué pasó, con qué datos, justo antes de un fallo — sin tener que reproducirlo en directo.
- **Auditar el comportamiento**: qué operaciones se han ejecutado, cuándo y con qué resultado.
- **Analizar patrones**: cuántas pólizas se han procesado, cuántas rechazado, cuánto ha tardado cada paso.

Una primera aproximación (insuficiente) sería usar `print()`:

In [1]:
nombre_archivo = "ejemplo.txt"
with open(nombre_archivo, "r", encoding="utf-8") as f:
    print(f"Abriendo archivo {nombre_archivo}")
    contenido = f.read()
print(f"Archivo {nombre_archivo} cerrado")

Abriendo archivo ejemplo.txt
Archivo ejemplo.txt cerrado


`print()` manda todo a `stdout` (la salida estándar). Si el programa falla, el mensaje de error va a `stderr` (la salida de error) — pero en ambos casos, en cuanto el proceso termina, ese texto **desaparece**. No hay marca de tiempo, no hay forma de filtrar por gravedad, no queda ningún registro persistente.

> ⚠️ **Error típico**: dejar en el código de producción `print()` de depuración ("aquí llega", "valor de x: ...") en vez de logging. Funcionan mientras miras la consola en directo, pero no sirven de nada para diagnosticar un fallo que ocurrió esta madrugada en un proceso batch sin nadie mirando la pantalla — es exactamente lo que revisa el checklist de commit del notebook de Git de este mismo día.

## 2. El módulo `logging`

`logging` es el módulo estándar de Python para gestionar logs — no requiere instalar nada. La forma más simple de configurarlo es con `logging.basicConfig()`:

In [2]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
)

logger = logging.getLogger(__name__)
logger.debug("Mensaje de depuración")

2026-09-07 08:45:56,880 | DEBUG    | __main__ | Mensaje de depuración


El formato usado (`%(asctime)s | %(levelname)-8s | %(name)s | %(message)s`) muestra:

- `asctime`: la marca de tiempo.
- `levelname`: el nivel de gravedad (`DEBUG`, `INFO`...).
- `name`: el *logger* que emitió el mensaje — por convención, `__name__` del módulo, para saber de dónde viene cada log.
- `message`: el mensaje en sí.

### Niveles de logging

El módulo estándar define cinco niveles (de menor a mayor gravedad):

| Nivel | Valor numérico | Uso típico |
|---|---|---|
| `DEBUG` | 10 | Detalle fino para depurar durante el desarrollo. |
| `INFO` | 20 | Funcionamiento normal ("póliza 1234 procesada correctamente"). |
| `WARNING` | 30 | Algo inusual que no impide seguir, pero merece revisión. |
| `ERROR` | 40 | Un fallo concreto que ha afectado a una operación. |
| `CRITICAL` | 50 | Un fallo que compromete el funcionamiento general de la aplicación. |

`level=logging.DEBUG` en `basicConfig` significa "muestra este nivel y todos los superiores". Si configuraras `level=logging.WARNING`, los `debug()` e `info()` quedarían silenciados.

In [3]:
logger.debug("Mensaje de depuración detallado")
logger.info("Póliza 1024 procesada correctamente")
logger.warning("Prima inusualmente alta para el perfil de riesgo")
logger.error("No se ha podido calcular la prima: dato de siniestralidad ausente")
logger.critical("Conexión con el motor de tarificación perdida")

2026-09-07 08:45:56,884 | DEBUG    | __main__ | Mensaje de depuración detallado


2026-09-07 08:45:56,885 | INFO     | __main__ | Póliza 1024 procesada correctamente


2026-09-07 08:45:56,885 | WARNING  | __main__ | Prima inusualmente alta para el perfil de riesgo


2026-09-07 08:45:56,885 | ERROR    | __main__ | No se ha podido calcular la prima: dato de siniestralidad ausente


2026-09-07 08:45:56,885 | CRITICAL | __main__ | Conexión con el motor de tarificación perdida


> **Nota**: si vienes de mirar Loguru (una librería de terceros muy popular) verás que allí hay siete niveles, incluyendo `TRACE` y `SUCCESS`. El módulo estándar `logging` solo tiene estos cinco — es su forma de mantenerse simple. Más sobre Loguru al final del notebook.

## 3. `getLogger(__name__)` en vez del logger raíz

En el ejemplo anterior usamos `logging.getLogger(__name__)`. En un notebook, `__name__` vale `"__main__"`; en un módulo real (`tarificacion.py`), valdría `"tarificacion"`. Usar un logger con nombre (en vez de llamar directamente a `logging.debug(...)`, el logger raíz) es la práctica estándar: permite identificar de qué parte del proyecto viene cada mensaje, y ajustar el nivel de detalle módulo a módulo si hace falta.

> 📊 En Excel esto sería como una columna de auditoría (`quién`, `cuándo`, `qué`) que se va rellenando sola en una hoja de registro, en vez de fiarte de la memoria de quién tocó qué celda y cuándo.

## 4. Formato personalizado y contexto adicional

El formato admite cualquier combinación de los [atributos disponibles](https://docs.python.org/3/library/logging.html#logrecord-attributes) (`asctime`, `levelname`, `name`, `filename`, `lineno`, `message`...). Para incluir datos concretos del negocio (por ejemplo, el número de póliza que se está procesando), la forma más simple es interpolarlos directamente en el mensaje con un f-string:

In [4]:
class Estimador:
    """Estima la prima de una póliza a partir de un factor de riesgo simplificado."""

    def __init__(self) -> None:
        self.logger = logging.getLogger(f"{__name__}.Estimador")

    def predict(self, numero_poliza: int, x: float) -> float:
        y = x * 0.1 + 3
        self.logger.info(
            "Predicción generada: poliza=%s, x=%.2f, y=%.2f", numero_poliza, x, y
        )
        return y


estimador = Estimador()
estimador.predict(numero_poliza=1024, x=5.0)
estimador.predict(numero_poliza=1025, x=8.0)

2026-09-07 08:45:56,890 | INFO     | __main__.Estimador | Predicción generada: poliza=1024, x=5.00, y=3.50


2026-09-07 08:45:56,890 | INFO     | __main__.Estimador | Predicción generada: poliza=1025, x=8.00, y=3.80


3.8

> ⚠️ **Error típico**: construir el mensaje con un f-string siempre, aunque el nivel de logging esté desactivado (`logger.debug(f"...")` con muchísimos datos, incluso si `DEBUG` no se está registrando). Usar el estilo `logger.info("... %s ...", valor)` (como arriba) hace que Python solo formatee el mensaje si realmente se va a emitir — más eficiente cuando el mensaje es costoso de construir (por ejemplo, si incluye volcar un DataFrame grande).

## 5. Registrar excepciones con contexto

Cuando capturas una excepción, es buena práctica loguearla con `logger.error(..., exc_info=True)` (o directamente `logger.exception(...)` dentro de un bloque `except`, que hace lo mismo pero de forma más breve). Esto añade el *traceback* completo al log — igual de específico que ver el error en pantalla, pero persistido.

In [5]:
def procesar_poliza(numero_poliza: int, edad: int) -> None:
    logger_proceso = logging.getLogger(f"{__name__}.procesar_poliza")
    try:
        if edad <= 0:
            raise ValueError(f"Edad inválida ({edad}) para la póliza {numero_poliza}")
        logger_proceso.info("Póliza %s procesada (edad=%s)", numero_poliza, edad)
    except ValueError:
        logger_proceso.exception("Fallo al procesar la póliza %s", numero_poliza)


procesar_poliza(numero_poliza=2001, edad=40)
procesar_poliza(numero_poliza=2002, edad=-5)

2026-09-07 08:45:56,896 | INFO     | __main__.procesar_poliza | Póliza 2001 procesada (edad=40)


2026-09-07 08:45:56,897 | ERROR    | __main__.procesar_poliza | Fallo al procesar la póliza 2002
Traceback (most recent call last):
  File "C:\Users\Arnau Ivern\AppData\Local\Temp\ipykernel_25236\2725082636.py", line 5, in procesar_poliza
    raise ValueError(f"Edad inválida ({edad}) para la póliza {numero_poliza}")
ValueError: Edad inválida (-5) para la póliza 2002


Fíjate en que `procesar_poliza(2002, -5)` no ha detenido el notebook: la excepción se ha capturado, registrado con todo su *traceback*, y el programa ha podido continuar. Esto es exactamente lo que quieres en un proceso batch que procesa miles de pólizas — que un dato inválido no tumbe todo el proceso, pero que quede constancia clara de qué falló y por qué.

**Para ti:** añade un `logger.warning(...)` dentro de `procesar_poliza` que se dispare cuando `edad > 90` (edad válida, pero inusual y digna de revisión manual), sin lanzar ninguna excepción. Pruébalo con `numero_poliza=2003, edad=95`.

In [6]:
# Tu código aquí


## 6. Guardar logs en un archivo

Además de mostrar los logs por consola, se puede (y en un proceso desatendido, se debe) guardar en un archivo con un `FileHandler`. Para no dejar artefactos sueltos dentro del repositorio del curso, este ejemplo escribe el archivo de log en una carpeta temporal del sistema (con el módulo `tempfile`), igual que hará el notebook de Git más adelante — en un proyecto real, ese archivo iría en una ruta de logs propia del proyecto (y normalmente excluida de git).

In [7]:
import tempfile
from pathlib import Path

directorio_logs = Path(tempfile.mkdtemp(prefix="curso_python_logging_"))
ruta_log = directorio_logs / "tarificacion.log"

logger_archivo = logging.getLogger("tarificacion_batch")
logger_archivo.setLevel(logging.INFO)

handler_archivo = logging.FileHandler(ruta_log, encoding="utf-8")
handler_archivo.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")
)
logger_archivo.addHandler(handler_archivo)

logger_archivo.info("Inicio del proceso de tarificación batch")
logger_archivo.info("Pólizas procesadas: 128, rechazadas: 3")
logger_archivo.warning("2 pólizas con prima fuera de rango habitual")

handler_archivo.close()  # aseguramos que el contenido se ha volcado a disco antes de leerlo

print("Contenido del archivo de log:")
print(ruta_log.read_text(encoding="utf-8"))

2026-09-07 08:45:56,906 | INFO     | tarificacion_batch | Inicio del proceso de tarificación batch


2026-09-07 08:45:56,906 | INFO     | tarificacion_batch | Pólizas procesadas: 128, rechazadas: 3


2026-09-07 08:45:56,907 | WARNING  | tarificacion_batch | 2 pólizas con prima fuera de rango habitual


Contenido del archivo de log:
2026-09-07 08:45:56,906 | INFO     | Inicio del proceso de tarificación batch
2026-09-07 08:45:56,906 | INFO     | Pólizas procesadas: 128, rechazadas: 3
2026-09-07 08:45:56,907 | WARNING  | 2 pólizas con prima fuera de rango habitual



Un `Handler` decide **a dónde** van los logs (consola, archivo, ambos a la vez); el `Formatter` decide **cómo se ven**. Un mismo logger puede tener varios handlers con formatos distintos — por ejemplo, un formato compacto en consola y uno más detallado en el archivo.

## 7. Nota opcional: Loguru

El módulo estándar `logging` es potente pero verboso de configurar (como has visto, `basicConfig` + `getLogger` + `Handler` + `Formatter`). Existen alternativas de terceros pensadas para simplificar la configuración; la más popular es [Loguru](https://github.com/Delgan/loguru), que ofrece un `logger` preconfigurado de fábrica, colores en consola, y una sintaxis más compacta para guardar en archivo o capturar excepciones automáticamente (`@logger.catch`).

No se enseña en detalle en este curso — el módulo estándar es la base que necesitas conocer y que encontrarás en cualquier proyecto Python, con o sin librerías de terceros — pero si en tu trabajo profesional montas un proyecto nuevo desde cero, es una opción legítima a valorar.

## Resumen

- `print()` no sirve como sistema de logging: no persiste, no tiene niveles ni marca de tiempo, y desaparece en cuanto el proceso termina.
- `logging.basicConfig(level=..., format=...)` configura el comportamiento por defecto del logging en un script o notebook.
- Los cinco niveles estándar (`DEBUG` < `INFO` < `WARNING` < `ERROR` < `CRITICAL`) filtran qué se registra según gravedad.
- Usa `logging.getLogger(__name__)` en vez del logger raíz, para poder identificar de qué módulo viene cada mensaje.
- Un `Handler` decide el destino de los logs (consola, archivo...); un `Formatter` decide su formato — un logger puede tener varios handlers a la vez.
- `logger.exception(...)` (o `logger.error(..., exc_info=True)`) dentro de un `except` registra el *traceback* completo, con contexto suficiente para depurar después sin tener que reproducir el fallo en directo.
- Loguru es una alternativa de terceros más ergonómica, mencionada aquí solo como referencia — el módulo estándar es la base de este curso.

**Siguiente:** `04_teoria_pytest.ipynb`, donde veremos cómo automatizar la verificación de que el código funciona con `pytest`.